# CP07 - Data pipeline BISINDO A-Z
Membaca split beku, memeriksa leakage, mengecek loader, dan menyimpan `results/sanity_summary.json`.
Jalankan dari folder `notebooks/` atau root repo (Run All).

In [ ]:
import sys, json, hashlib, os
from pathlib import Path
import yaml, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import dataset as ds

cfg = yaml.safe_load((ROOT / "configs/config.yaml").read_text())
SEED = cfg["seed"]
np.random.seed(SEED)

# config.yaml menyebut split_v1.csv; file yang sekarang ada adalah splits.csv
SPLIT_PATH = ROOT / cfg["split_path"]
if not SPLIT_PATH.exists():
    SPLIT_PATH = ROOT / "splits/splits.csv"
LABEL_MAP_PATH = ROOT / cfg["label_map_path"]
RESULTS = ROOT / cfg["results_dir"]
DATA_ROOT = ROOT / "data/BISINDO Indonesian Sign Language Alphabet Image Data"
HAS_IMAGES = DATA_ROOT.exists()
print("split:", SPLIT_PATH.relative_to(ROOT), "| gambar tersedia:", HAS_IMAGES)

## 1. Baca split dan label map

In [ ]:
df = pd.read_csv(SPLIT_PATH)
need = {"path", "label", "label_idx", "split"}
assert need <= set(df.columns), f"kolom kurang: {need - set(df.columns)}"
assert set(df["split"]) == {"train", "val", "test"}

# label map: satu-satunya sumber pemetaan A-Z -> 0-25
LABEL_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
LABEL_MAP_PATH.write_text(json.dumps(ds.CLASS_TO_IDX, indent=2))
assert (df["label"].map(ds.CLASS_TO_IDX) == df["label_idx"]).all()

print(len(df), "gambar")
print(df["split"].value_counts(normalize=True).round(3).to_string())
df.head()

## 2. Cek leakage

In [ ]:
checks = {}

# path ganda
checks["path_unik"] = not df["path"].duplicated().any()

# grup tidak muncul di lebih dari satu split
g_nsplit = df.groupby("group")["split"].nunique()
checks["grup_tidak_lintas_split"] = bool((g_nsplit == 1).all())

# 26 huruf ada di tiap split
checks["26_huruf_per_split"] = bool((df.groupby("split")["label"].nunique() == 26).all())

# duplikat exact (md5) antar split, dari results/sanitary/duplicates_exact.csv
dup = pd.read_csv(RESULTS / "sanitary/duplicates_exact.csv")
sp = df.set_index("path")["split"]
dup["split"] = dup["path"].map(sp)
dup_in = dup.dropna(subset=["split"])
cross = dup_in.groupby("md5")["split"].nunique()
checks["md5_tidak_lintas_split"] = bool((cross <= 1).all())
n_md5_leak = int((cross > 1).sum())

# kelompok subjek/latar: kosong = keterbatasan
checks["subject_terdeteksi"] = bool(df["subject"].notna().mean() > 0.5) if "subject" in df else False
print(json.dumps(checks, indent=2))
print("md5 yang tersebar di >1 split:", n_md5_leak)
dup_in[dup_in["md5"].isin(cross[cross > 1].index)] if n_md5_leak else None

Catatan: `group` di split ini adalah `source_id` (foto yang sama), bukan orang atau latar.
Jadi orang/latar yang sama masih bisa muncul di train dan test. Ini keterbatasan yang harus ditulis di README.

## 3. Jumlah gambar per kelas per split

In [ ]:
ct = pd.crosstab(df["label"], df["split"])[["train", "val", "test"]]
display(ct.T)
ct.plot(kind="bar", stacked=True, figsize=(12, 4), title="Gambar per kelas per split")
plt.ylabel("jumlah"); plt.tight_layout()
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(RESULTS / "figures/class_counts_per_split.png", dpi=150)
plt.show()
ct.to_csv(RESULTS / "class_counts_per_split.csv")

## 4. Grid contoh gambar per huruf (train/val/test)

In [ ]:
if HAS_IMAGES:
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(6, 5, figsize=(12, 15))
    for ax, letter in zip(axes.ravel(), ds.CLASSES):
        sub = df[df["label"] == letter]
        split = ["train", "val", "test"][ds.CLASS_TO_IDX[letter] % 3]
        row = sub[sub["split"] == split].iloc[rng.integers(len(sub[sub["split"] == split]))]
        ax.imshow(Image.open(DATA_ROOT / row["path"]).convert("RGB"))
        ax.set_title(f"{letter} ({row['split']})"); ax.axis("off")
    for ax in axes.ravel()[26:]: ax.axis("off")
    plt.tight_layout()
    plt.savefig(RESULTS / "figures/sample_grid.png", dpi=120)
    plt.show()
else:
    print("Data mentah tidak ditemukan, lewati.")

## 5. Loader: satu batch

In [ ]:
batch_info = None
if HAS_IMAGES:
    loaders, class_w, _ = ds.get_dataloaders(
        DATA_ROOT, batch_size=32, splits_csv=SPLIT_PATH, num_workers=0, seed=SEED)
    # augmentasi hanya train
    for s in ("train", "val", "test"):
        print(s, [type(t).__name__ for t in loaders[s].dataset.transform.transforms])
    x, y = next(iter(loaders["train"]))
    batch_info = dict(shape=list(x.shape), min=float(x.min()), max=float(x.max()),
                      mean=float(x.mean()), labels_range=[int(y.min()), int(y.max())])
    print(batch_info)
    assert x.shape[1:] == (3, cfg["img_size"], cfg["img_size"])
    assert 0 <= y.min() and y.max() < 26
else:
    print("Data mentah tidak ditemukan, lewati.")

## 6. Ringkasan sanity file (dari `sanitary_report.json`)

In [ ]:
rep = json.loads((RESULTS / "sanitary/sanitary_report.json").read_text())
for k in ("total_files", "corrupt_or_invalid", "issue_breakdown", "per_variant",
          "image_modes", "imbalance_ratio_max_min", "resolution"):
    print(k, ":", rep.get(k))

## 7. Simpan `results/sanity_summary.json`

In [ ]:
leakage_free = all(checks[k] for k in ("path_unik", "grup_tidak_lintas_split", "md5_tidak_lintas_split"))
summary = dict(
    seed=SEED, split_file=str(SPLIT_PATH.relative_to(ROOT)),
    n_images=int(len(df)),
    split_counts=df["split"].value_counts().to_dict(),
    split_fraction=df["split"].value_counts(normalize=True).round(4).to_dict(),
    checks=checks, md5_groups_across_splits=n_md5_leak,
    leakage_free=bool(leakage_free),
    batch=batch_info,
    corrupt_or_invalid=rep.get("corrupt_or_invalid"),
    note="group = source_id; orang/latar yang sama masih bisa lintas split",
)
(RESULTS / "sanity_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 8. Kenapa preprocessing dipisah dari augmentasi
Preprocessing (resize, normalisasi) harus identik di train, val, dan test supaya input model konsisten.
Augmentasi mengubah gambar secara acak, hanya untuk train agar model lebih tahan variasi. Jika diterapkan pada val/test, skor jadi tidak stabil dan tidak mewakili data nyata.